# Stage 1 - Non-Instruction Fine-Tuning (Continued Pretraining)

**Goal:** teach the base model the *language and facts* of the client e-commerce schema
by continuing pretraining on the raw schema corpus.

This adapts the model to domain vocabulary (table names like `X_Product_Images`, columns,
relationships) **before** we teach it to follow instructions in Stage 2.

Pipeline: **Base -> [Stage 1] -> Stage 2 (SFT) -> Stage 3 (DPO)**

Data and adapters both live on the Hugging Face Hub.

In [ ]:
# Install Unsloth (needs a CUDA GPU). If Colab prompts to restart after install, do it.
%%capture
!pip install --upgrade unsloth unsloth_zoo
# Ensure a MODERN TRL (provides SFTConfig/DPOConfig + processing_class) that matches new transformers.
!pip install --upgrade "trl>=0.13.1"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > Change runtime type > T4 GPU")

## 1. Config + load the raw domain corpus from Hugging Face

In [ ]:
# --- Datasets live on the Hugging Face Hub (pushed via scripts/push_to_hf.py) ---
HF_USER = "Rajesh507"   # <-- your Hugging Face username
DS_NONINSTRUCT = f"{HF_USER}/ecomm-db-noninstruct"
DS_INSTRUCTION = f"{HF_USER}/ecomm-db-instruction"
DS_PREFERENCE  = f"{HF_USER}/ecomm-db-preference"

# If the datasets are PRIVATE, log in first (needs a read token):
# from huggingface_hub import login; login()

In [ ]:
# Persist trained adapters to the Hugging Face Hub (no Google Drive needed).
from huggingface_hub import login
login()   # paste a WRITE token: https://huggingface.co/settings/tokens

ADAPTER_STAGE1 = f"{HF_USER}/ecomm-db-stage1-noninstruct"
ADAPTER_STAGE2 = f"{HF_USER}/ecomm-db-stage2-sft"
ADAPTER_STAGE3 = f"{HF_USER}/ecomm-db-stage3-dpo"
print("Adapters will be pushed under:", HF_USER)

In [ ]:
from datasets import load_dataset
raw = load_dataset(DS_NONINSTRUCT, split="train")   # column 'text' = one block per row
blocks = [b for b in raw["text"] if b and b.strip()]
print("Blocks:", len(blocks))
print("Example:\n", blocks[0][:300])

## 2. Group blocks into training chunks

In [ ]:
chunks, cur = [], ""
for b in blocks:
    if len(cur) + len(b) < 800:
        cur += ("\n\n" + b) if cur else b
    else:
        if cur:
            chunks.append(cur)
        cur = b
if cur:
    chunks.append(cur)
print("Chunks:", len(chunks))

## 3. Load base model with Unsloth (4-bit QLoRA)

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B"   # Coder variant is stronger at SQL. Alt: "unsloth/Qwen2.5-1.5B"
MAX_SEQ_LEN = 2048

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
EOS = tokenizer.eos_token
ds = Dataset.from_dict({"text": [c + EOS for c in chunks]})
print(ds)

## 4. Apply LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                       # LoRA rank
    lora_alpha = 16,              # scaling
    lora_dropout = 0,             # 0 is optimized in Unsloth
    bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Train on the raw text

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,   # new TRL API (was 'tokenizer=')
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LEN,
        dataset_num_proc = 2,
        packing = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 10,      # more passes so the domain vocabulary actually sticks
        learning_rate = 5e-5,       # lower LR for continued pretraining
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage1",
        report_to = "none",
    ),
)
trainer_stats = trainer.train()

## 6. Push the Stage-1 adapter to the Hugging Face Hub

In [ ]:
model.push_to_hub(ADAPTER_STAGE1, token=True)
tokenizer.push_to_hub(ADAPTER_STAGE1, token=True)
print("Pushed Stage-1 adapter to:", ADAPTER_STAGE1)

## 7. Quick sanity test (completion style, not Q&A yet)

In [ ]:
FastLanguageModel.for_inference(model)
prompt = "The X_Product_Images table stores"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=60, temperature=0.3)
print(tokenizer.decode(out[0], skip_special_tokens=True))

### Next
Proceed to `instruction_finetuning.ipynb`.